这是我使用 NVIDIA 的 NEMOTRON-3-NANO-OMINI-30B-A3B 模型的第一周项目...这是一个推理模型，就像 GPT-5 一样，用于计算统计数据的熵。

In [29]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
import math
from binance.client import Client
import numpy as np
import pandas as pd
from scipy.stats import entropy
import requests
import json
import os
from dotenv import load_dotenv
from openai import OpenAI

基本上这段代码是从 binance 获取 1 分钟蜡烛数据，你不需要任何 API 密钥 
为了这个目的......

In [30]:
# 币安客户端
client = Client()

# 拿蜡烛
klines = client.get_klines(
    symbol='BTCUSDT',
    interval=Client.KLINE_INTERVAL_1MINUTE,
    limit=500
)

In [31]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 数据框
df = pd.DataFrame(klines, columns=[
    'open_time', 'open', 'high', 'low', 'close', 'volume',
    'close_time', 'quote_asset_volume', 'num_trades',
    'taker_buy_base', 'taker_buy_quote', 'ignore'
])
print(df)

         open_time            open            high             low  \
0    1778309040000  80238.57000000  80238.58000000  80222.13000000   
1    1778309100000  80222.14000000  80222.14000000  80208.33000000   
2    1778309160000  80217.35000000  80219.65000000  80210.35000000   
3    1778309220000  80219.65000000  80281.13000000  80219.64000000   
4    1778309280000  80261.53000000  80270.00000000  80253.62000000   
..             ...             ...             ...             ...   
495  1778338740000  80332.23000000  80332.23000000  80332.22000000   
496  1778338800000  80332.23000000  80332.23000000  80321.23000000   
497  1778338860000  80321.23000000  80330.21000000  80320.00000000   
498  1778338920000  80330.21000000  80349.40000000  80330.20000000   
499  1778338980000  80349.40000000  80349.40000000  80349.39000000   

              close       volume     close_time quote_asset_volume  \
0    80222.14000000   3.32787000  1778309099999    266987.89462700   
1    80217.35000000

In [32]:
# 转换收盘价
df['close'] = df['close'].astype(float)

# 计算日志返回
df['returns'] = np.log(df['close'] / df['close'].shift(1))

# 删除 NaN
returns = df['returns'].dropna()

# 创建直方图箱
hist, bins = np.histogram(returns, bins=20)

# 转换为概率
probabilities = hist / hist.sum()

# 消除零概率
probabilities = probabilities[probabilities > 0]

# 手动香农熵
shannon_entropy = -sum(
    p * math.log2(p)
    for p in probabilities
)

h=entropy(probabilities,base=2)

print("Shannon Entropy:", shannon_entropy)
print("Shannon Entropy using scipy:", h)

close = df["close"].astype(float)

# 日志返回
returns = np.log(close / close.shift(1)).dropna()

# 方差
variance = np.var(returns)

# 高斯熵
G = 0.5 * math.log2(
    2 * math.pi * math.e * variance
)

print("Gaussian Entropy:", G)

Shannon Entropy: 2.752202013798435
Shannon Entropy using scipy: 2.7522020137984344
Gaussian Entropy: -10.242077636891816


既然我们已经得到了香农熵，那么是时候使用 NVIDIA 的 AI 模型来分析它了
在这里，我们使用 scipy.stats 和纯香农熵公式计算了香农熵

我们使用高斯熵和香农熵来理解市场结构

In [33]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
load_dotenv(override=True)
api_key=os.getenv("OPENROUTER_API_KEY")
client=OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

这里我们将使用 API 调用与 openrouter 进行通信
首先，我们将使用端点向 openrouter API 发送 POST 请求

In [34]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 首次 API 调用并进行推理
response = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  headers={
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
  },
  data=json.dumps({
    "model": "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    "messages": [
{
  "role": "system",
  "content": """
You are a quantitative market analyst.

Your task:
- Interpret Shannon entropy and Gaussian entropy
- Analyze market structure
- Detect regime conditions
- Explain whether the market is:
  - trending
  - ranging
  - volatile
  - compressing
  - chaotic

Rules:
- NEVER reveal internal reasoning
- NEVER explain your thinking process
- ONLY provide the final market interpretation
- Keep response structured and professional
"""
},
      {
  "role": "user",
  "content": f"""
Market entropy metrics:

Shannon Entropy: {shannon_entropy}

Gaussian Entropy: {G}

Interpret:
- structural randomness
- volatility randomness
- probable market regime
- trend strength
- market stability
- possible upcoming behavior

Respond like a professional quant analyst.
"""
}
      ],
    "reasoning": {"enabled": True}
  })
)

In [35]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
response = response.json()
response = response['choices'][0]['message']['content']
print(response)

**Market Interpretation**

- **Structural Randomness:** Moderate (Shannon entropy ≈ 2.75) – the market exhibits a blend of deterministic patterns and stochastic fluctuations, but not extreme chaos.  
- **Volatility Randomness:** Low (Gaussian entropy ≈ ‑10.24) – indicates compressed, low‑variance price movements, suggesting a regime of reduced volatility dispersion.  
- **Probable Market Regime:** Ranging/compressing – the combined entropy signals a stable, non‑trending environment where price action oscillates within a confined band.  
- **Trend Strength:** Weak – limited directional persistence is inferred from the entropy levels, implying any existing trend is fragile.  
- **Market Stability:** High – the negative Gaussian entropy points to a relatively stable price landscape with minimal abrupt swings.  
- **Possible Upcoming Behavior:** Expect continuation of range‑bound activity; a sustained move would require a rise in entropy (increased randomness) or a sharp expansion in volat

基于熵的市场结构分析

该项目探索使用香农熵和高斯熵作为定量工具，利用币安的实时市场数据分析金融市场结构和制度行为
。这种方法不是仅仅依赖传统的技术指标，而是将市场视为一个信息动态系统，其中的随机性、波动性和结构组织可以通过熵理论进行数学测量。该系统获取历史蜡烛数据，计算对数回报，并评估价格变动的信息属性，以了解市场是否有组织、混乱、压缩、扩张、趋势或范围。

香农熵用于衡量市场内的结构不确定性。它通过分析离散市场状态或回报分布的概率分布来评估价格行为的组织性或随机性。较低的香农熵通常表明结构组织增加、方向持久性和更强的趋势行为，而较高的香农熵表明随机性、噪声、优柔寡断和范围限制条件。在此框架中，香农熵充当信息混乱的衡量标准，有助于识别市场运动是否包含可利用的结构或由随机行为主导。

高斯熵用于衡量连续财务回报分布中的波动不确定性和离散度。与在符号或离散状态上运行的香农熵不同，高斯熵直接从市场回报的方差导出，反映了市场的活力状态。较高的高斯熵通常对应于波动性扩张、不稳定的市场行为、清算事件或爆炸性运动，而较低的高斯熵则代表波动性压缩、安静的积累阶段和市场能量减少。由于财务回报在短窗口内通常近似高斯分布，因此高斯熵提供了市场波动结构的有效连续信息表示。

香农熵和高斯熵的结合创建了市场行为的多维视图。通过分析结构随机性和波动性随机性，该系统可以比传统指标更有效地对市场状况进行分类。例如，低香农熵与高斯熵相结合可能表明强烈的突破或扩张趋势阶段，而高香农熵与低高斯熵相结合可能代表低效的低能斩波或横盘运动。这种基于熵的状态分类框架允许系统研究金融市场中的相变，类似于物理学中的热力学系统，其中熵代表无序，波动性代表市场能量。

该实现使用 Python 以及 NumPy、pandas、requests 和 math 等库来获取和处理市场数据。从 Binance API 检索历史蜡烛数据，转换为返回序列，然后转换为概率分布以进行熵计算。

该项目作为熵驱动的量化金融系统的基础研究框架。未来的扩展可能包括滚动熵分析、熵导数、转移熵、条件熵、谱熵、蒙特卡洛模拟、傅里叶域分析、政权转换建模、博弈论市场动态和自适应算法交易系统。更广泛的目标是建立一个信息论市场情报引擎，能够通过数学分析而不是传统的基于指标的启发法来识别隐藏的结构变化和不断变化的市场状态。

香农熵是使用经典信息理论公式手动实现的：H(X)=−Σi=1n​p(xi​)log2​p(xi​)
高斯熵是使用连续收益分布的方差计算的： H=1/2​log2​(2πeσ^2)